# 02 Reconciliation gate

Before anything is derived from these tables, I rebuild the ONS's own published headline from them and
confirm it matches at the precision the ONS published. The gate raises an error if any blocking check fails,
so a scripted run stops here and the chart notebook never runs.

Blocking checks cover the headline percentages, the counts, the significance results, the wording in the
committed bulletin snapshot and the structure of the two tables. Informational checks record things I found
along the way. They don't stop the run, but each one is written to `reconciliation.csv`.

**What these figures are.** Each estimate is the share of people aged 16 and over (or 16 to 59) in
England and Wales who told the Crime Survey for England and Wales they had experienced domestic abuse
in the previous 12 months. They describe millions of people's experience, and I treat them that way.

**The ONS's assessment, which I hold every table and label to.** The year ending March 2026 increase
*includes real change*. Changes to the survey at the start of that year likely affected its size, by an
amount that cannot yet be measured. Nothing in this project estimates that amount, and nothing here
should be read as suggesting the increase is only a survey effect.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import (PAGES, PAGE_SOURCES, POPULATION, PROCESSED, RAW, RETRIEVED, SOURCES, TABLES,
                    page_text, population_16_plus, read_sheet, row, sha256)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
table1 = pd.read_parquet(PROCESSED / "table1_new_questions.parquet")
table2 = pd.read_parquet(PROCESSED / "table2_old_questions.parquet")
AST = TABLES / "annualsupplementarytables_yemar2026.xlsx"
DAPVC25 = TABLES / "domesticabuseprevalenceandvictimcharacteristics_yemar2025.xlsx"
DAPVC23 = TABLES / "domesticabuseprevalenceandvictimcharacteristics_yemar2023.xlsx"
bulletin = page_text("ons_bulletin_crime_ew_yejune2026.html")

results = []


def check(name, passed, expected, got, blocking=True, note=""):
    results.append(dict(check=name, blocking=blocking, passed=bool(passed),
                        expected=str(expected), got=str(got), note=note))
    flag = "PASS" if passed else ("FAIL" if blocking else "NOTE")
    print(f"[{flag}] {name}: expected {expected}, got {got}")


def est(year_ending, measure):
    s = table1[(table1.year_ending == year_ending) & (table1.measure == measure)]
    assert len(s) == 1
    return s.iloc[0]

## 1. Headline percentages at published precision

The ONS publishes these to one decimal place in the bulletin. I round the unrounded table values once, at
the point of comparison, and nowhere else.

In [ ]:
PUBLISHED = {("2026-03", "any domestic abuse"): 9.6, ("2026-03", "any partner abuse"): 7.9,
             ("2026-03", "any family abuse"): 2.8, ("2025-03", "any domestic abuse"): 7.8,
             ("2025-03", "any partner abuse"): 6.1, ("2025-03", "any family abuse"): 2.7}
for (ye, m), published in PUBLISHED.items():
    got = est(ye, m).estimate_pct
    check(f"{m}, YE {ye}, rounds to bulletin figure", round(got, 1) == published, published, f"{got:.4f}")

# The partner and family figures are not parts of the total: some people experience both, so they overlap.
# I show that rather than let anyone try to rebuild 9.6% by adding them.
p, f, a = (est("2026-03", m).estimate_pct for m in ("any partner abuse", "any family abuse", "any domestic abuse"))
check("partner + family exceeds any domestic abuse (overlap, so no sum-of-parts rebuild)", p + f > a,
      f"> {a:.2f}", f"{p + f:.2f}", blocking=False)

## 2. The two YE March 2026 sheets agree, and earlier releases agree with this one

A9a (with intervals) and A9c (time series) are separate sheets in one workbook, so I check they carry the same
YE March 2026 numbers. Then I compare YE March 2024 and 2025 with the figures first published in the YE March 2025
edition, to see whether anything was revised between releases.

In [ ]:
a9a = read_sheet(AST, "Table A9a", "Type of abuse")
a9c = read_sheet(AST, "Table A9c", "Type of abuse")
c26 = next(c for c in a9c.columns if c.startswith("Apr 2025 to Mar 2026 All"))
for label in ("Any domestic abuse", "Any partner abuse", "Any family abuse"):
    x, y = float(row(a9a, label)["All in the last year Estimate"]), float(row(a9c, label)[c26])
    check(f"A9a and A9c agree for {label.lower()}, YE 2026-03", x == y, f"{x:.10f}", f"{y:.10f}")

t2_25 = read_sheet(DAPVC25, "Table 2", "Type of abuse")
for period, ye in (("Apr 2023 to Mar 2024", "2024-03"), ("Apr 2024 to Mar 2025", "2025-03")):
    first = float(row(t2_25, "Any domestic abuse")[next(c for c in t2_25.columns if c.startswith(f"{period} All"))])
    now = est(ye, "any domestic abuse").estimate_pct
    check(f"any domestic abuse YE {ye}: unchanged since the YE March 2025 edition", abs(first - now) < 1e-9,
          f"{first:.10f}", f"{now:.10f}", blocking=False)

## 3. The ONS's significance tests match what the bulletin says

The bulletin says the overall and partner increases are significant and that family abuse shows no significant
difference. A9c carries the test result for each.

In [ ]:
for m, expected in (("any domestic abuse", "[s]"), ("any partner abuse", "[s]"), ("any family abuse", "[ns]")):
    got = est("2026-03", m).change_vs_previous_year
    check(f"{m}: YE 2026-03 vs 2025-03 significance flag", got == expected, expected, got)

## 4. Counts

The ONS's note says counts are "derived by multiplying the prevalence rate by the population aged 16 years and
over in England and Wales" from the mid-year estimates. I tried that with the published mid-year estimates, and it
does not reproduce the ONS's own published counts. Section 4b records that. So I rebuild the counts from the
ONS's own published rate and count pairs instead:

* **YE March 2025**: the count is published directly (Table 3 of the YE March 2025 edition), so I check it rounds to "around 3.8 million".
* **YE March 2026**: no table publishes a domestic abuse count; it appears only in the bulletin text. Worksheet A8 of
  the same release publishes a rate *and* a count for the combined measure, on the same population base (A8 note 5).
  That pair gives the population the ONS used, and I apply it to the domestic abuse rate. It is a different measure,
  so the rebuild is not circular.

In [ ]:
t3_25 = read_sheet(DAPVC25, "Table 3", "Type of abuse")
any25 = t3_25[(t3_25.iloc[:, 0] == "Any domestic abuse") & (t3_25.Sex == "All")].iloc[0]
count25_k = float(any25["Victims in the last year (000s)"])
check("YE 2025-03 count rounds to 'around 3.8 million'", round(count25_k / 1000, 1) == 3.8, 3.8, f"{count25_k:,.0f}k")

wb_rows = read_sheet(AST, "Worksheet A8", "Abuse type")   # first table in the sheet is A8a (rates)
a8_rate = float(wb_rows.iloc[0, 1])
a8_count_k = float(wb_rows[wb_rows.iloc[:, 0].astype(str).str.startswith("Any domestic abuse")].iloc[1, 1])
denominator_k = a8_count_k / (a8_rate / 100)
# The A8 count is rounded to the nearest thousand, so the population it implies has a small range.
denom_range = [(a8_count_k + d) / (a8_rate / 100) for d in (-0.5, 0.5)]
rate26 = est("2026-03", "any domestic abuse").estimate_pct
count26_k = rate26 / 100 * denominator_k
count_range = [rate26 / 100 * d for d in denom_range]
check("YE 2026-03 count rebuilt from A8's population rounds to 'around 4.7 million'",
      all(round(c / 1000, 1) == 4.7 for c in count_range), 4.7,
      f"{count26_k:,.0f}k (range {count_range[0]:,.0f}k to {count_range[1]:,.0f}k)",
      note=f"population implied by A8: {denominator_k:,.0f}k aged 16+")

denominator25_k = count25_k / (est("2025-03", "any domestic abuse").estimate_pct / 100)
print(f"\nPopulation implied by the ONS's own counts: YE 2025-03 {denominator25_k:,.0f}k, YE 2026-03 {denominator_k:,.0f}k")
print(f"Difference between the two published counts: {count26_k - count25_k:,.0f}k, of which about "
      f"{rate26 / 100 * (denominator_k - denominator25_k):,.0f}k comes from the larger population base alone")

### 4b. What the published mid-year estimates give (informational)

I record this, but it doesn't block the gate. Summing single years of age 16 and over from MYE2 gives a population
about 2 million larger than the one the ONS's own counts imply. The ONS's counts are probably based on a different
population (for example, the household population the survey samples), but I haven't confirmed that. It's a
question for crimestatistics@ons.gov.uk, and the article should quote the ONS's counts, not this method.

In [ ]:
pop24, title24 = population_16_plus(POPULATION / "mye24tablesew.xlsx")
pop25, title25 = population_16_plus(POPULATION / "mye25tablesew.xlsx")
mye25_k = est("2025-03", "any domestic abuse").estimate_pct / 100 * pop24 / 1000
mye26_k = rate26 / 100 * pop25 / 1000
check("MYE mid-2024 16+ method reproduces the YE 2025-03 published count", abs(mye25_k - count25_k) < 1,
      f"{count25_k:,.0f}k", f"{mye25_k:,.0f}k (population {pop24:,})", blocking=False,
      note="documented discrepancy: ask ONS which population base the counts use")
check("MYE mid-2025 16+ method reproduces 'around 4.7 million'", round(mye26_k / 1000, 1) == 4.7, 4.7,
      f"{mye26_k:,.0f}k (population {pop25:,})", blocking=False,
      note="documented discrepancy: ask ONS which population base the counts use")

## 5. The bulletin snapshot says what the tables say

These are the exact phrases in the committed snapshot of the bulletin (retrieved 31 October 2026). I also keep
the ONS's assessment word for word in `ons_wording.csv`, so any label or caption quotes it from the snapshot
rather than from memory.

In [ ]:
PHRASES = {
    "headline 9.6% and count": "Around 9.6% of people aged 16 years and over experienced domestic abuse in the last "
                               "year (around 4.7 million people)",
    "previous year 7.8% and count": "(7.8%; around 3.8 million people)",
    "partner abuse 7.9% vs 6.1%": "significant increase in partner abuse (7.9%) compared with the previous year (6.1%)",
    "family abuse 2.8% vs 2.7%": "(2.8% in YE March 2026 compared with 2.7% in YE March 2025)",
    "ONS assessment: real change": "our best assessment is that this increase also includes real change rather than "
                                   "solely being the result of sampling uncertainty and question changes",
    "ONS assessment: survey changes": "Changes to the questionnaire and its implementation at the start of the survey "
                                      "year have likely affected the scale of the increase in our prevalence estimates",
    "ONS assessment: too early": "it is too early to say whether this increase will come to represent a new, higher "
                                "level of prevalence",
}
for name, phrase in PHRASES.items():
    check(f"bulletin snapshot contains: {name}", phrase in bulletin, "present", "present" if phrase in bulletin else "missing")

wording = pd.DataFrame([{"key": k, "text": v, "source": "ons_bulletin_crime_ew_yejune2026.html", "retrieved": RETRIEVED}
                        for k, v in PHRASES.items() if k.startswith("ONS assessment")])
wording.to_csv(PROCESSED / "ons_wording.csv", index=False)

## 6. The two bases stay apart

The break is the point of this piece, so the gate checks the tables can't be joined by accident: separate questions
bases, one age band per long run, the old series ending at YE March 2024 and its two gaps kept as gaps. I also
compare the old series between the YE March 2023 and 2024 editions, to check for revisions (informational).

In [ ]:
check("questions bases are disjoint between the tables",
      set(table1.questions_basis).isdisjoint(table2.questions_basis), "disjoint",
      sorted(set(table1.questions_basis) | set(table2.questions_basis)))
long_run = table2[table2.long_run_series & table2.measure.eq("any domestic abuse")]
check("long-run old series is ages 16 to 59 only", set(long_run.age_band) == {"16 to 59"}, {"16 to 59"}, set(long_run.age_band))
check("long-run old series runs YE 2005-03 to YE 2024-03",
      (long_run.year_ending.min(), long_run.year_ending.max()) == ("2005-03", "2024-03"),
      ("2005-03", "2024-03"), (long_run.year_ending.min(), long_run.year_ending.max()))
check("YE 2008-03 and 2021-03 kept as gaps with no estimate",
      long_run.loc[long_run.estimate_pct.isna(), "year_ending"].tolist() == ["2008-03", "2021-03"],
      ["2008-03", "2021-03"], long_run.loc[long_run.estimate_pct.isna(), "year_ending"].tolist())
last_old = long_run.dropna(subset=["estimate_pct"]).iloc[-1]
check("last old-questions estimate (16 to 59, YE 2024-03) as published", round(last_old.estimate_pct, 2) == 5.35,
      5.35, f"{last_old.estimate_pct:.4f}")

t4_23 = read_sheet(DAPVC23, "Table 4", "Type of abuse")
r23 = row(t4_23, "Any domestic abuse (")
diffs = []
for c in t4_23.columns[1:]:
    if " compared with " in c or not isinstance(r23[c], (int, float)):
        continue
    ye = f"{c.split(' ')[4]}-03"
    now = long_run.loc[long_run.year_ending == ye, "estimate_pct"]
    if len(now):
        diffs.append(abs(float(r23[c]) - float(now.iloc[0])))
check("old series unchanged between the YE 2023 and YE 2024 editions", max(diffs) < 1e-9, 0, f"max diff {max(diffs):.2e}",
      blocking=False)

## Verdict

In [ ]:
recon = pd.DataFrame(results)
recon.to_csv(PROCESSED / "reconciliation.csv", index=False)
recon.to_parquet(PROCESSED / "reconciliation.parquet", index=False)

headline = pd.DataFrame([
    dict(figure="any domestic abuse, YE March 2026", published="9.6%", rebuilt=f"{rate26:.4f}%"),
    dict(figure="any domestic abuse, YE March 2025", published="7.8%",
         rebuilt=f"{est('2025-03', 'any domestic abuse').estimate_pct:.4f}%"),
    dict(figure="people, YE March 2026", published="around 4.7 million", rebuilt=f"{count26_k:,.0f}k"),
    dict(figure="people, YE March 2025", published="around 3.8 million", rebuilt=f"{count25_k:,.0f}k"),
])
headline.to_csv(PROCESSED / "headline_rebuilt.csv", index=False)

failed = recon[recon.blocking & ~recon.passed]
notes = recon[~recon.blocking & ~recon.passed]
print(f"{recon.blocking.sum()} blocking checks, {len(failed)} failed; "
      f"{(~recon.blocking).sum()} informational, {len(notes)} recorded as notes")
if len(failed):
    raise RuntimeError("Reconciliation gate FAILED:\n" + failed[["check", "expected", "got"]].to_string(index=False))

# The chart notebook reads this file and checks the table hashes, so it can't run against tables the gate didn't see.
status = {"gate": "passed", "retrieved": RETRIEVED,
          "inputs": {n: sha256(PROCESSED / f"{n}.parquet") for n in ("table1_new_questions", "table2_old_questions")}}
(PROCESSED / "gate_status.json").write_text(json.dumps(status, indent=2))
print("Reconciliation gate PASSED")
headline